# Installation

In [ ]:
!pip install -q unsloth==2025.5.7
!pip install -q SPARQLWrapper
!pip install -q langchain-community

# Load Dataset

In [ ]:
!wget -O train_entity_prop.json -nv https://raw.githubusercontent.com/gansixeneh/FROG-2.0/refs/heads/dataset/dataset/labels/qald_9_plus_train_wikidata_converted_labels.json
!wget -O train_sparql.json -nv https://raw.githubusercontent.com/gansixeneh/FROG-2.0/refs/heads/dataset/dataset/possible_uris/train_cot.json

In [ ]:
import json
import pandas as pd

def get_dataset_sparql(path_to_data: str):
    with open(path_to_data, "r", encoding="utf-8") as file:
        raw = json.load(file)
        
    data = []
    for item in raw:
        data.append({
            "question": item['question'],
            "entities_matches": item['entities_matches'],
            "properties_matches": item['properties_matches'],
            "sparql": item['sparql'],
            "thoughts": item.get('thoughts', None)
        })
    
    df = pd.DataFrame(data)
    print(f"Extracted {len(df)} data from {path_to_data}")

    return df

def get_dataset_entity_prop(path_to_data: str):
    with open(path_to_data, "r", encoding="utf-8") as file:
        raw = json.load(file)
        
    data = []
    for item in raw:
        data.append({
            "question": item['question'],
            "entities": item['entities'],
            "properties": item['properties'],
        })
    
    df = pd.DataFrame(data)
    print(f"Extracted {len(df)} data from {path_to_data}")

    return df


train_sparql = get_dataset_sparql("train_sparql.json")
train_entity_prop = get_dataset_entity_prop("train_entity_prop.json")

# Model

In [ ]:
from unsloth import FastLanguageModel
import torch
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
from datasets import Dataset
from typing import Optional
from torch.nn.parallel import DataParallel
import gc
from huggingface_hub import HfApi, create_repo
import shutil

model_name = "mistralai/Mistral-Nemo-Instruct-2407"
save_to_gguf = False
hf_token = "MY_HF_TOKEN"

max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

def load_model(model_name: str):
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_name,
        max_seq_length=max_seq_length,
        dtype=dtype,
        load_in_4bit=load_in_4bit,
    )
        
    # Set the Qwen2.5 chat template
    if not hasattr(tokenizer, "chat_template") and tokenizer.chat_template is not None and 'qwen' in model.config.name_or_path:
        tokenizer.chat_template = "<|im_start|>system\n{{ messages[0]['content'] }}<|im_end|>\n{% for message in messages[1:] %}<|im_start|>{{ message['role'] }}\n{{ message['content'] }}<|im_end|>\n{% endfor %}"
        print("Model's tokenizer has been set.")
    
    return model, tokenizer

def finetune_sparql(model_name, train_dataset, val_dataset, output_dir, num_epochs=10):
    """
    Input adapter_path if you want to save your adapter weights
    """    
    model, tokenizer = load_model(model_name)

    model = FastLanguageModel.get_peft_model(
        model,
        r = 64,  # Increased from 32 for more capacity to handle structured syntax
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                         "gate_proj", "up_proj", "down_proj",],
        lora_alpha = 64,  # Matching r
        lora_dropout = 0.1,  # Increased for better generalization 
        bias = "all",  # Changed from "none" to better learn syntax patterns
        use_gradient_checkpointing = "unsloth",
        random_state = 42,
        use_rslora = True,  # Keep RSLoRA for stability
        loftq_config = None,
    )
    
    trainer = SFTTrainer(
        model = model,
        tokenizer = tokenizer,
        train_dataset = train_dataset,
        eval_dataset = val_dataset,
        dataset_text_field = "text",
        max_seq_length = max_seq_length,
        dataset_num_proc = 4,  # Increased for faster processing
        packing = True,  # Enable packing for efficiency with varied sequence lengths
        args = TrainingArguments(
            # Batch sizing
            per_device_train_batch_size = 1,  # Reduced to focus on quality over speed
            per_device_eval_batch_size = 1,
            gradient_accumulation_steps = 16,  # Increased for larger effective batch
            
            # Learning rate
            learning_rate = 1e-5,  # Lower learning rate for careful convergence
            weight_decay = 0.1,  # Increased regularization to combat overfitting
            
            # Schedule
            num_train_epochs = num_epochs,
            # max_steps = 2,
            lr_scheduler_type = "polynomial",  # Better than cosine for syntax learning
            warmup_ratio = 0.1,  # Longer warmup (10% of training) 
            
            # Precision
            fp16 = not is_bfloat16_supported(),
            bf16 = is_bfloat16_supported(),
            optim = "adamw_8bit",
            
            # Evaluation
            eval_strategy = "steps",
            save_strategy = "steps",
            eval_steps = 25,  # More frequent evaluation
            save_steps = 25,
            logging_steps = 5,
            
            # Checkpoint management
            save_total_limit = 5,  # Keep more checkpoints
            load_best_model_at_end = True,
            metric_for_best_model = "eval_loss",
            greater_is_better = False,
            
            # Additional settings
            group_by_length = True,
            gradient_checkpointing = True,
            dataloader_drop_last = False,
            
            # Add label smoothing for SPARQL syntax tokens
            label_smoothing_factor = 0.05,
            
            # Output
            output_dir = "outputs",
            report_to = "tensorboard",
            seed = 42,
        ),
    )
    
    trainer.train()

    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)

    if save_to_gguf:
        save_repo_name = f"gansixeneh/{model_name.split('/')[-1]}"
        repo_name = f"{save_repo_name}-sparql"
        create_repo(repo_name, token=hf_token, exist_ok=True)
        model.push_to_hub_gguf(repo_name, tokenizer, quantization_method = "q4_k_m", token=hf_token)
        shutil.rmtree('/kaggle/working/gansixeneh')
    
    return model, tokenizer

def finetune_entity_prop(model_name, train_dataset, val_dataset, output_dir, num_epochs=5):
    """
    Fine-tune a model with LoRA
    """
    model, tokenizer = load_model(model_name)
    
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                         "gate_proj", "up_proj", "down_proj"],
        lora_alpha=16,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=42,
        use_rslora=False,
        loftq_config=None,
    )
    
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        dataset_text_field="text",
        max_seq_length=max_seq_length,
        dataset_num_proc=2,
        packing=False,
        args=TrainingArguments(
            per_device_train_batch_size=1,
            per_device_eval_batch_size=1,
            gradient_accumulation_steps=4,
            warmup_steps=5,
            num_train_epochs=num_epochs,
            # max_steps = 2,
            learning_rate=2e-5,
            fp16=not is_bfloat16_supported(),
            bf16=is_bfloat16_supported(),
            optim="adamw_8bit",
            weight_decay=0.01,
            lr_scheduler_type="linear",
            seed=42,
            output_dir="outputs",
            report_to="none",
            
            eval_strategy="steps",
            save_strategy="steps",
            save_steps=20,
            eval_steps=20,
            logging_steps=10,
            
            # Checkpoint management
            save_total_limit=5,
            load_best_model_at_end=True,
            metric_for_best_model="eval_loss",
            greater_is_better=False,
        ),
    )

    trainer.train()

    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    
    if save_to_gguf:
        save_repo_name = f"gansixeneh/{model_name.split('/')[-1]}"
        repo_name = f"{save_repo_name}-extract-entity"
        create_repo(repo_name, token=hf_token, exist_ok=True)
        model.push_to_hub_gguf(repo_name, tokenizer, quantization_method = "q4_k_m", token=hf_token)
        shutil.rmtree('/kaggle/working/gansixeneh')
    
    return model, tokenizer

In [ ]:
model, tokenizer = load_model(model_name)

# Finetune SPARQL

In [ ]:
has_chat_template = hasattr(tokenizer, "chat_template") and tokenizer.chat_template is not None
has_chat_template

In [ ]:
# Split train val
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(train_sparql, test_size=0.2, random_state=42)

In [ ]:
from typing import List, Dict, Any, Literal

# Format data for instruction tuning
system_prompt = """You are a SPARQL generator expert for Wikidata knowledge graph. Your task is to convert the following natural language question to a SPARQL query for Wikidata using the provided entity and property resolutions.

Guidelines:
1. First identify which entities from the list match the question's intent
2. Identify which entities are relevant to the question and select EXACTLY ONE entity ID for each distinct concept in the question
3. When multiple entities have similar labels, choose the one whose description best matches the question's context
4. From the properties list, choose which properties are needed to answer the question
5. Select only the minimum necessary properties required to answer the question correctly
6. Use ALL identified entities and necessary properties in your SPARQL query
7. Use PREFIX NOTATION ONLY (e.g., wd:Q123, wdt:P123), NOT full URIs
8. Optimize your query by using appropriate SPARQL features (DISTINCT, FILTER, ORDER BY, LIMIT) when needed
9. Return entity IDs directly without using label services
10. Return ONLY the raw SPARQL query with no explanations or comments in this format:
   ```sparql
   <your_sparql_query_here>
   ```

IMPORTANT: Before generating the SPARQL query, provide your step-by-step reasoning about how to construct the query.
"""

user_prompt_template = """Question: {question}

Entities:
{entities_matches}

Properties:
{properties_matches}

SPARQL:"""

# Updated assistant prompt template to include thoughts
asst_prompt_template = """{thoughts}

```sparql
{sparql}
```"""

def format_entity_matches(matches: List[Dict[str, Any]]):
    result = ""
    for item, matches in matches.items():
        for match in matches:
            result += f"- id: {match['id']}, label: {match['label']}, description: {match['description']}\n"
    return result[:-2]  # Remove last extra line

def format_property_matches(matches: List[Dict[str, Any]]):
    result = ""
    for item, matches in matches.items():
        for match in matches:
            result += f"- id: {match['id']}, label: {match['label']}, description: {match['description']}\n"
    return result[:-2]  # Remove last extra line

def format_instruction(row):
    entities_matches_formatted = format_entity_matches(row['entities_matches'])
    properties_matches_formatted = format_property_matches(row['properties_matches'])

    user_prompt = user_prompt_template.format(question=row['question'],
                                              entities_matches=entities_matches_formatted,
                                              properties_matches=properties_matches_formatted)
    
    # Format thoughts to be more readable in the assistant's response
    thoughts_formatted = "Let me analyze this step by step:\n\n"
    for i, thought in enumerate(row['thoughts'], 1):
        thoughts_formatted += f"{thought}\n"
    
    asst_prompt = asst_prompt_template.format(thoughts=thoughts_formatted,
                                             sparql=row['sparql'])

    if has_chat_template:
        prompt = tokenizer.apply_chat_template(
            [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
                {"role": "assistant", "content": asst_prompt}
            ],
            tokenize=False,
            add_generation_prompt=False,
            continue_final_message=False
        )
    else:
        prompt = system_prompt + '\n' + user_prompt + '\n' + asst_prompt
    
    return prompt

# Format data for Unsloth
train_df['text'] = train_df.apply(format_instruction, axis=1)
val_df['text'] = val_df.apply(format_instruction, axis=1)
train_dataset = Dataset.from_pandas(train_df[['text']])
val_dataset = Dataset.from_pandas(val_df[['text']])

In [ ]:
print(train_df['text'].iloc[0])

In [ ]:
del model
gc.collect()
torch.cuda.empty_cache()

In [ ]:
ft_model, ft_tokenizer = finetune_sparql(model_name, train_dataset, val_dataset, "ft_model_sparql")
!tar -czvf ft_model_sparql.tar.gz ft_model_sparql/

In [ ]:
del ft_model, ft_tokenizer
gc.collect()
torch.cuda.empty_cache()

# Finetune Extract Entity & Property

In [ ]:
# Split train val
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(train_entity_prop, test_size=0.2, random_state=42)

In [ ]:
from typing import List, Dict, Any

# Format data for entity and property extraction
system_prompt = """You are an expert entity and property extractor for knowledge graph querying. Your task is to analyze a natural language question and identify the relevant entities and properties needed to create a SPARQL query for Wikidata.

Guidelines:
1. For each question, extract ALL entities mentioned in the question
2. For each question, extract ALL relevant properties needed to answer the question
3. Format your response as a structured JSON object with 'entities' and 'properties' keys
4. Each key should contain an array of strings with the entity or property names
5. Focus ONLY on extracting, not on generating SPARQL queries

Your output should look like:
```json
{
  "entities": ["entity1", "entity2", ...],
  "properties": ["property1", "property2", ...]
}
```
"""

user_prompt_template = """Question: {question}

Extract all entities and properties from this question that would be needed to generate a SPARQL query for Wikidata."""

asst_prompt_template = """```json
{{
  "entities": {entities},
  "properties": {properties}
}}
```"""

def format_instruction(row):
    # Format entities and properties as JSON arrays
    entities_json = json.dumps(row['entities'])
    properties_json = json.dumps(row['properties'])
    
    user_prompt = user_prompt_template.format(question=row['question'])
    asst_prompt = asst_prompt_template.format(
        entities=entities_json,
        properties=properties_json
    )

    if has_chat_template:
        prompt = tokenizer.apply_chat_template(
            [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
                {"role": "assistant", "content": asst_prompt}
            ],
            tokenize=False,
            add_generation_prompt=False,
            continue_final_message=False
        )
    else:
        prompt = system_prompt + '\n' + user_prompt + '\n' + asst_prompt
    
    return prompt

# Format data for Unsloth
train_df['text'] = train_df.apply(format_instruction, axis=1)
val_df['text'] = val_df.apply(format_instruction, axis=1)
train_dataset = Dataset.from_pandas(train_df[['text']])
val_dataset = Dataset.from_pandas(val_df[['text']])

In [ ]:
print(train_df['text'].iloc[0])

In [ ]:
ft_model, ft_tokenizer = finetune_entity_prop(model_name, train_dataset, val_dataset, "ft_model_entity_prop")
!tar -czvf ft_model_entity_prop.tar.gz ft_model_entity_prop/